# Part 1: Text Classification Representation Learning (Wikipedia Articles)

In this notebook, we dynamically scrape real Wikipedia articles across different concepts to build our custom dataset! We will train an MLP, snapshot the latent space embeddings across 11 stages (Epoch 0 through Epoch 10), and output them to a single Mantis-ready CSV file.


In [14]:
# Install required dependencies
!pip install torch torchvision pandas numpy scikit-learn umap-learn bokeh

In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import numpy as np
import urllib.request
import urllib.parse
import json
import os
from sklearn.model_selection import train_test_split
import umap
from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.palettes import Category10
from bokeh.layouts import gridplot

output_notebook()


Loading BokehJS ...

## 1. Fetch the Wikipedia Dataset
We define a function that searches Wikipedia for multiple topics and returns the introductory extracts of the top 100 pages for each topic. You can change these topics to absolutely anything your class enjoys!


In [16]:
def fetch_wiki_search(query, limit=250):
    print(f"Fetching wikipedia articles for: {query}...")
    url = f"https://en.wikipedia.org/w/api.php?action=query&list=search&srsearch={urllib.parse.quote(query)}&srlimit={limit}&format=json"
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 (MIT Python Lab 6.390)'})
    response = urllib.request.urlopen(req)
    data = json.loads(response.read())
    pages = data['query']['search']
    
    pageids = [str(p['pageid']) for p in pages]
    docs = []
    
    for i in range(0, len(pageids), 50):
        batch = "|".join(pageids[i:i+50])
        ext_url = f"https://en.wikipedia.org/w/api.php?action=query&prop=extracts&exintro=true&explaintext=true&pageids={batch}&format=json"
        ext_req = urllib.request.Request(ext_url, headers={'User-Agent': 'Mozilla/5.0 (MIT Python Lab)'})
        ext_response = urllib.request.urlopen(ext_req)
        ext_data = json.loads(ext_response.read())
        pages_ext = ext_data['query']['pages']
        for pid, pdata in pages_ext.items():
            extract = pdata.get('extract', '')
            if extract.strip():
                title_clean = "".join(c for c in pdata.get('title','') if c.isalnum() or c in (' ', '_')).rstrip()
                docs.append({'Title': pdata.get('title',''), 'Description': extract, 'Category': query})
                
                # Save the article to a local folder for fun viewing!
                folder_path = os.path.join("wikipedia_articles", query.replace(" ", "_"))
                os.makedirs(folder_path, exist_ok=True)
                with open(os.path.join(folder_path, f"{title_clean}.txt"), "w", encoding="utf-8") as text_file:
                    text_file.write(f"Title: {pdata.get('title','')}\n\n{extract}")
    return docs

topics = ["Machine learning", "Molecular biology", "Renaissance art", "Quantum physics", "Basketball", "Pure mathematics"]
all_docs = []
for t in topics:
    all_docs.extend(fetch_wiki_search(t, limit=150))

df_data = pd.DataFrame(all_docs)
print(f"\nFetched {len(df_data)} total articles!")

# Prepare TF-IDF features based exclusively on the Wikipedia abstract extract
vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
X = vectorizer.fit_transform(df_data['Description']).toarray()

# Map categories to indices
category_to_idx = {cat: idx for idx, cat in enumerate(topics)}
y = df_data['Category'].map(category_to_idx).values

X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.long)


Fetching wikipedia articles for: Machine learning...
Fetching wikipedia articles for: Molecular biology...
Fetching wikipedia articles for: Renaissance art...
Fetching wikipedia articles for: Quantum physics...
Fetching wikipedia articles for: Basketball...
Fetching wikipedia articles for: Pure mathematics...

Fetched 360 total articles!


## 2. Model Architecture
We track representations right before the ultimate `classifier` layer via the `self.embeddings` hook.


In [17]:
# We define a simple Multi-Layer Perceptron (MLP) for text classification.
class TextClassifier(nn.Module):
    def __init__(self, vocab_size, hidden_dim, num_classes):
        super(TextClassifier, self).__init__()
        self.fc1 = nn.Linear(vocab_size, 256)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(256, hidden_dim) # Bottleneck representation
        self.classifier = nn.Linear(hidden_dim, num_classes)
        self.embeddings = None # This will act as our 'hook' to save the latent features
        
    def forward(self, x):
        out = self.relu(self.fc1(x))
        out = self.fc2(out)
        self.embeddings = out.detach().cpu().numpy() # Save point
        out = self.relu(out)
        out = self.classifier(out)
        return out

model = TextClassifier(vocab_size=1000, hidden_dim=64, num_classes=len(topics))
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.002)


## 3. Train & Capture Representations over 11 Epochs
Snapshot Epoch 0 (Initialization) and then do the standard gradient descent for 10 epochs, grabbing embeddings along the way.


In [18]:
epochs = 10
embeddings_per_epoch = []

# Epoch 0 (Initialization)
model.eval()
with torch.no_grad():
    model(X_tensor)
    # Save a copy of the latent representation to track how it evolves!
    embeddings_per_epoch.append(model.embeddings.copy())
model.train()

print("Training...")
# ---- Main Training Loop ----
# We iterate over the dataset multiple times (epochs)
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(X_tensor)
    loss = criterion(outputs, y_tensor)
    loss.backward()
    optimizer.step()
    
    acc = (outputs.argmax(dim=1) == y_tensor).float().mean()
    if (epoch + 1) % 2 == 0 or epoch == 0:
        print(f"Epoch [{(epoch+1):2d}/{epochs}], Loss: {loss.item():.4f}, Accuracy: {acc.item():.4f}")
    
    model.eval()
    with torch.no_grad():
        model(X_tensor)
        # Save a copy of the latent representation to track how it evolves!
        embeddings_per_epoch.append(model.embeddings.copy())
    model.train()

print(f"\nCaptured embeddings spanning {len(embeddings_per_epoch)} states.")


Training...
Epoch [ 1/10], Loss: 1.7935, Accuracy: 0.1667
Epoch [ 2/10], Loss: 1.7834, Accuracy: 0.1667
Epoch [ 4/10], Loss: 1.7579, Accuracy: 0.1917
Epoch [ 6/10], Loss: 1.7185, Accuracy: 0.5750
Epoch [ 8/10], Loss: 1.6608, Accuracy: 0.9611
Epoch [10/10], Loss: 1.5811, Accuracy: 0.9944

Captured embeddings spanning 11 states.


## 4. Export the Unified CSV
By formatting the output table with embedding_epoch_0, embedding_epoch_1, etc., Mantis can dynamically parse the multi-step training trajectory in one shot.


In [19]:
df_out = pd.DataFrame({
    'title': df_data['Title'],
    'categoric': df_data['Category'],
    'article_snippet': df_data['Description'].apply(lambda x: x[:150] + "...")
})

for i, embs in enumerate(embeddings_per_epoch):
    df_out[f'embedding_epoch_{i}'] = embs.tolist()

csv_name = 'text_analysis_wikipedia_mantis.csv'
df_out.to_csv(csv_name, index=False)
print(f"Saved {csv_name}!")


Saved text_analysis_wikipedia_mantis.csv!


## 5. Notebook Visualization grid (UMAP)
To visualize locally, we perform UMAP for all 11 epochs individually and place them in a time series grid. You should observe the data initially clustered randomly, then gradually organizing into structural islands!


In [20]:
from bokeh.io import output_notebook
output_notebook()
plots = []
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)

colors = Category10[max(3, len(topics))]
color_map = {cat: color for cat, color in zip(topics, colors)}
df_out['color'] = df_out['categoric'].map(color_map)

print("Generating UMAP plots. This will take a moment...")
for i, embs in enumerate(embeddings_per_epoch):
    umap_coords = reducer.fit_transform(embs)
    
    plot_df = pd.DataFrame({
        'coordinate1': umap_coords[:, 0],
        'coordinate2': umap_coords[:, 1],
        'color': df_out['color'],
        'categoric': df_out['categoric'],
        'title': df_out['title']
    })
    
    source = ColumnDataSource(plot_df)
    p = figure(title=f"Epoch {i}", width=300, height=300)
    p.scatter('coordinate1', 'coordinate2', size=4, color='color', legend_group='categoric', alpha=0.7, source=source)
    p.axis.visible = False
    if i != 0:
        p.legend.visible = False
        
    hover = HoverTool(tooltips=[("Category", "@categoric"), ("Title", "@title")])
    p.add_tools(hover)
    plots.append(p)

show(gridplot(plots, ncols=3))


Loading BokehJS ...

Generating UMAP plots. This will take a moment...


/Users/adityasengupta/miniconda3/envs/juliaenv/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
